# Bonus — 3-Stage Rocket Mission to ISS

**Classical Mechanics · Project I · EETAC, UPC**

Full ECI simulation of a Soyuz-type 3-stage rocket reaching the ISS (408 km orbit).
Includes gravity turn, staging, Hohmann transfer, circularisation burn and rendezvous.

> ⚠️ This simulation takes ~10–15 seconds to run.

In [ ]:
import urllib.request, os
BASE = "https://raw.githubusercontent.com/gdlcjoel-lab/PROYECTOMECANICA/main/Definitivos/"
for f in ["bonus_multistage.py", "bonus_plot.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + f, f)
        print(f"Downloaded {f}")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline
print("Setup complete ✓")

In [ ]:
from bonus_multistage import simulate_multistage, print_mission_summary

print("Running 3-stage simulation... (this takes ~10 seconds)")
data = simulate_multistage(dt_atm=0.1, dt_space=5.0)
print_mission_summary(data)

## Plots
Mission timeline (altitude, speed, mass) and 2-D ECI orbital view.

In [ ]:
import numpy as np
import matplotlib.patches as mpatches

t     = data["t"]
h     = data["h"]
speed = data["speed"]
m     = data["m"]
phase = data["phase"]
r     = data["r"]

PHASE_COLORS = {1:"#5B9BD5",2:"#2A6496",3:"#0D2B55",4:"#B07D2A",5:"#8E44AD",6:"#27AE60"}
PHASE_NAMES  = {1:"Stage 1",2:"Stage 2",3:"Coast",4:"Circularisation",5:"Hohmann transfer",6:"ISS orbit"}

# ── Figure 1: Timeline ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("3-Stage Rocket Mission — Timeline", fontsize=14, fontweight="bold", color="#0D2B55")
fig.patch.set_facecolor("white")
axes = axes.flatten()

for p in np.unique(phase):
    mask = phase == p
    col  = PHASE_COLORS[p]
    axes[0].plot(t[mask], h[mask],     ".", color=col, markersize=1)
    axes[1].plot(t[mask], speed[mask], ".", color=col, markersize=1)
    axes[2].plot(t[mask], m[mask],     ".", color=col, markersize=1)

for ev_t, _ in data["events"]:
    for ax in axes[:3]:
        ax.axvline(ev_t, color="#aaa", linewidth=0.6, linestyle=":")

axes[0].axhline(408, color="#27AE60", linewidth=1.2, linestyle="--", label="ISS (408 km)")
axes[0].legend(fontsize=8)

for ax, (title, xl, yl) in zip(axes[:3], [
    ("Altitude vs Time", "Time [s]", "Altitude [km]"),
    ("Speed vs Time",    "Time [s]", "Speed [m/s]"),
    ("Mass vs Time",     "Time [s]", "Mass [kg]"),
]):
    ax.set_title(title, fontweight="bold", color="#0D2B55")
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.grid(True, alpha=0.4)

# Phase portrait
for p in np.unique(phase):
    mask = phase == p
    axes[3].plot(speed[mask], h[mask], ".", color=PHASE_COLORS[p], markersize=1, label=PHASE_NAMES[p])
axes[3].set_title("Altitude vs Speed (phase portrait)", fontweight="bold", color="#0D2B55")
axes[3].set_xlabel("Speed [m/s]"); axes[3].set_ylabel("Altitude [km]"); axes[3].grid(True, alpha=0.4)

legend_patches = [mpatches.Patch(color=PHASE_COLORS[p], label=PHASE_NAMES[p]) for p in sorted(PHASE_COLORS)]
fig.legend(handles=legend_patches, loc="lower center", ncol=6, fontsize=8, bbox_to_anchor=(0.5, 0))
plt.tight_layout(rect=[0,0.06,1,1])
plt.savefig("bonus_timeline.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Figure 2: Orbital view ─────────────────────────────────────────
from bonus_multistage import R_EARTH, H_ISS
theta = np.linspace(0, 2*np.pi, 360)
x_km = r[:,0]/1e3; y_km = r[:,1]/1e3

fig2, ax2 = plt.subplots(figsize=(8, 8))
fig2.patch.set_facecolor("white")
ax2.fill(R_EARTH/1e3*np.cos(theta), R_EARTH/1e3*np.sin(theta), color="#DDEEFF")
ax2.plot(R_EARTH/1e3*np.cos(theta), R_EARTH/1e3*np.sin(theta), color="#0D2B55", linewidth=1, label="Earth")
r_iss = (R_EARTH+H_ISS)/1e3
ax2.plot(r_iss*np.cos(theta), r_iss*np.sin(theta), color="#27AE60", linewidth=1.2, linestyle="--", label="ISS orbit (408 km)")
for p in np.unique(phase):
    mask = phase == p
    ax2.plot(x_km[mask], y_km[mask], ".", color=PHASE_COLORS[p], markersize=1, label=PHASE_NAMES[p])
ax2.plot(R_EARTH/1e3, 0, "*", color="#B07D2A", markersize=12, label="Launch site")
ax2.set_aspect("equal"); ax2.set_xlabel("ECI x [km]"); ax2.set_ylabel("ECI y [km]")
ax2.set_title("Orbital View — ECI x-y plane", fontweight="bold", color="#0D2B55")
ax2.legend(fontsize=8, loc="upper right"); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("bonus_orbit.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figures saved ✓")